## Modelo Escolhido — Métricas Completas

Recalcula a tabela `modelo_escolhido` com todas as métricas de regressão relevantes
para **todos os modelos candidatos** em cada nível, não só o vencedor.

| Métrica | O que mede | Interpretação |
|---|---|---|
| MAE | Erro absoluto médio | Quanto erra em unidades — fácil de explicar |
| Erro_% | MAE / média real × 100 | Erro relativo — comparável entre níveis |
| RMSE | Raiz do erro quadrático médio | Penaliza erros grandes — mais sensível a picos |
| R² | Variância explicada pelo modelo | 1.0 = perfeito, 0 = igual à média, < 0 = ruim |


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

engine = create_engine("sqlite:///../data/DBVendas.db")


In [2]:
# ==============================
# CARREGAR DADOS
# ==============================
print("📊 Carregando dados...")

query = """
SELECT
    pd.Data_Venda,
    iv.ID_Produto,
    pd.ID_Canal,
    iv.Qtde
FROM itens_vendas iv
JOIN vendas pd ON iv.ID_Pedido = pd.ID_Pedido
"""

df = pd.read_sql(query, engine)
df['Data_Venda'] = pd.to_datetime(df['Data_Venda'])

print(f"✅ Dados: {df.shape}")


📊 Carregando dados...
✅ Dados: (100000, 4)


In [3]:
# ==============================
# PREPARAÇÃO (sem leakage)
# ==============================

def preparar_dados(df, colunas):
    df_agg = df.groupby(colunas + ['Data_Venda'])['Qtde'].sum().reset_index()

    df_agg['Ano'] = df_agg['Data_Venda'].dt.year
    df_agg['Mes'] = df_agg['Data_Venda'].dt.month
    df_agg = df_agg.sort_values(colunas + ['Data_Venda'])

    df_agg['Lag_1'] = df_agg.groupby(colunas)['Qtde'].shift(1)
    df_agg['Lag_2'] = df_agg.groupby(colunas)['Qtde'].shift(2)
    df_agg['Lag_3'] = df_agg.groupby(colunas)['Qtde'].shift(3)

    df_agg['Media_3'] = (
        df_agg.groupby(colunas)['Qtde']
        .transform(lambda x: x.shift(1).rolling(3).mean())
    )
    df_agg['Media_6'] = (
        df_agg.groupby(colunas)['Qtde']
        .transform(lambda x: x.shift(1).rolling(6).mean())
    )

    df_agg['Mes_sin'] = np.sin(2 * np.pi * df_agg['Mes'] / 12)
    df_agg['Mes_cos'] = np.cos(2 * np.pi * df_agg['Mes'] / 12)

    df_agg = df_agg.dropna()
    return df_agg


In [4]:
# ==============================
# AVALIAÇÃO COM 4 MÉTRICAS
# ==============================

def avaliar_todos(train, test):
    """
    Treina todos os candidatos e retorna métricas completas para cada um.
    Retorna lista de dicts + o modelo vencedor (menor MAE).
    """
    X_train = train.drop(columns=['Qtde', 'Data_Venda'])
    y_train = train['Qtde']
    X_test  = test.drop(columns=['Qtde', 'Data_Venda'])
    y_test  = test['Qtde']

    candidatos = {
        "LinearRegression"       : LinearRegression(),
        "RandomForest"           : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "HistGradientBoosting"   : HistGradientBoostingRegressor(
                                        max_iter=200, learning_rate=0.05,
                                        max_depth=6, random_state=42),
    }

    resultados = []

    for nome, modelo in candidatos.items():
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        mae      = mean_absolute_error(y_test, y_pred)
        rmse     = np.sqrt(mean_squared_error(y_test, y_pred))
        r2       = r2_score(y_test, y_pred)
        erro_pct = (mae / y_test.mean()) * 100 if y_test.mean() > 0 else np.nan

        resultados.append({
            "nome"   : nome,
            "modelo" : modelo,
            "MAE"    : round(mae, 4),
            "Erro_%" : round(erro_pct, 4),
            "RMSE"   : round(rmse, 4),
            "R2"     : round(r2, 4),
        })

    resultados.sort(key=lambda x: x["MAE"])
    return resultados


In [5]:
# ==============================
# EXECUÇÃO POR NÍVEL
# ==============================

niveis = {
    "produto"      : ['ID_Produto'],
    "canal"        : ['ID_Canal'],
    "produto_canal": ['ID_Produto', 'ID_Canal'],
}

cutoff = '2021-01-01'

registros = []

for nome_nivel, colunas in niveis.items():

    print(f"\n🚀 Nível: {nome_nivel}")

    df_prep = preparar_dados(df.copy(), colunas)

    train = df_prep[df_prep['Data_Venda'] < cutoff]
    test  = df_prep[df_prep['Data_Venda'] >= cutoff]

    resultados = avaliar_todos(train, test)

    for i, r in enumerate(resultados):
        vencedor = (i == 0)
        print(f"  {'🏆' if vencedor else '  '} {r['nome']:<26} "
              f"MAE={r['MAE']:.2f}  Erro%={r['Erro_%']:.2f}  "
              f"RMSE={r['RMSE']:.2f}  R²={r['R2']:.4f}")

        registros.append({
            "Nivel"    : nome_nivel,
            "Modelo"   : r["nome"],
            "Vencedor" : vencedor,
            "MAE"      : r["MAE"],
            "Erro_%"   : r["Erro_%"],
            "RMSE"     : r["RMSE"],
            "R2"       : r["R2"],
            "N_Treino" : len(train),
            "N_Teste"  : len(test),
            "Data_Execucao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })



🚀 Nível: produto
  🏆 HistGradientBoosting       MAE=6.69  Erro%=48.54  RMSE=8.01  R²=0.0003
     LinearRegression           MAE=6.69  Erro%=48.56  RMSE=8.01  R²=-0.0002
     RandomForest               MAE=6.81  Erro%=49.41  RMSE=8.19  R²=-0.0453

🚀 Nível: canal
  🏆 LinearRegression           MAE=41.17  Erro%=27.75  RMSE=51.25  R²=-0.0044
     RandomForest               MAE=41.79  Erro%=28.17  RMSE=52.06  R²=-0.0364
     HistGradientBoosting       MAE=42.04  Erro%=28.34  RMSE=52.01  R²=-0.0343

🚀 Nível: produto_canal
  🏆 HistGradientBoosting       MAE=6.48  Erro%=48.37  RMSE=7.65  R²=0.0004
     LinearRegression           MAE=6.48  Erro%=48.37  RMSE=7.65  R²=0.0001
     RandomForest               MAE=6.53  Erro%=48.70  RMSE=7.75  R²=-0.0252


In [6]:
# ==============================
# SALVAR
# ==============================

df_modelo = pd.DataFrame(registros)

print("\n📊 Tabela modelo_escolhido:")
print(df_modelo.to_string(index=False))

df_modelo.to_sql("modelo_escolhido", engine, if_exists="replace", index=False)

print("\n✅ Tabela modelo_escolhido salva com métricas completas.")



📊 Tabela modelo_escolhido:
        Nivel               Modelo  Vencedor     MAE  Erro_%    RMSE      R2  N_Treino  N_Teste       Data_Execucao
      produto HistGradientBoosting      True  6.6869 48.5438  8.0110  0.0003     85815     7818 2026-04-14 16:56:58
      produto     LinearRegression     False  6.6891 48.5597  8.0130 -0.0002     85815     7818 2026-04-14 16:56:58
      produto         RandomForest     False  6.8060 49.4082  8.1918 -0.0453     85815     7818 2026-04-14 16:56:58
        canal     LinearRegression      True 41.1672 27.7524 51.2508 -0.0044      8022      726 2026-04-14 16:57:03
        canal         RandomForest     False 41.7910 28.1730 52.0586 -0.0364      8022      726 2026-04-14 16:57:03
        canal HistGradientBoosting     False 42.0444 28.3438 52.0082 -0.0343      8022      726 2026-04-14 16:57:03
produto_canal HistGradientBoosting      True  6.4801 48.3665  7.6516  0.0004     86819     8038 2026-04-14 16:57:43
produto_canal     LinearRegression     False

In [7]:
# ==============================
# RESUMO — VENCEDOR POR NÍVEL
# ==============================

vencedores = df_modelo[df_modelo['Vencedor']].copy()

print("\n🏆 Vencedor por nível:\n")
print(
    vencedores[['Nivel','Modelo','MAE','Erro_%','RMSE','R2']]
    .to_string(index=False)
)

print("""
Interpretação do R²:
  1.00  → modelo perfeito
  0.80+ → muito bom
  0.50+ → razoável
  < 0   → pior que usar a média histórica
""")



🏆 Vencedor por nível:

        Nivel               Modelo     MAE  Erro_%    RMSE      R2
      produto HistGradientBoosting  6.6869 48.5438  8.0110  0.0003
        canal     LinearRegression 41.1672 27.7524 51.2508 -0.0044
produto_canal HistGradientBoosting  6.4801 48.3665  7.6516  0.0004

Interpretação do R²:
  1.00  → modelo perfeito
  0.80+ → muito bom
  0.50+ → razoável
  < 0   → pior que usar a média histórica

